# 06 — Pathway-informed shrinkage: `group_horseshoe`

A generic version of "we already know a candidate gene set/pathway is relevant": each gene
belongs to a group (e.g. pathway membership), and the group gets a shared local scale, so a
group that collectively carries signal is shrunk less even if individual genes look modest
alone (the classic group-horseshoe construction).

This is directly motivated by comparing drug mechanisms that act through known pathways (e.g.
GIP vs. GLP-1 receptor signaling) — the informative group here stands in for "genes in the
mechanistically relevant pathway," kept generic. Since a real pathway annotation is never
perfect, the second half of this notebook corrupts the group (`perturb_group`) to see how much
misannotation the prior tolerates before losing its edge over the plain (unstructured)
horseshoe.

## Setup

In [ ]:
import numpy as np
import ptgs_bc as ptgs
from ptgs_bc import (simulate_twas_dataset, perturb_group, ElasticNetBuilder, BayesBuilder,
                     run_benchmark, summary_table, per_fold_table, plot_performance,
                     plot_comparison)
%matplotlib inline
print("ptgs_bc", ptgs.__version__)

## Simulate TWAS GReX with a known causal-gene set

In [ ]:
ds, true_w = simulate_twas_dataset(
    n_samples=200, n_genes=150, n_snps_per_gene=15, window_overlap=8, ld_rho=0.6,
    eqtl_sparsity=0.3, n_causal_genes=10, trait_pve=0.4, effect_sd=1.5, seed=0)
causal_idx = np.array(ds.meta["causal_idx"])
true_groups = np.zeros(ds.n_genes, dtype=int)
true_groups[causal_idx] = 1
print(f"p={ds.n_genes} genes, n={ds.n_samples} samples, "
      f"{len(causal_idx)} causal (the 'pathway')")

## Perfectly informative group vs. the plain horseshoe

The group here IS the true causal-gene set — the best case for `group_horseshoe`.

In [ ]:
builders_perfect = [
    ElasticNetBuilder(),
    BayesBuilder(prior="regularized_horseshoe", p0=10, num_warmup=300, num_samples=300),
    BayesBuilder(prior="group_horseshoe", p0=10, num_warmup=300, num_samples=300,
                 hyper={"groups": true_groups}),
]
res_perfect = run_benchmark(builders_perfect, ds, outer_k=5, seed=0)
summary_table(res_perfect)

In [ ]:
plot_comparison(res_perfect)

## A realistically imperfect pathway: how much misannotation is tolerable?

`perturb_group` drops some true causal genes from the group and adds some non-causal ones —
the two failure modes of a real pathway database.

In [ ]:
rng = np.random.default_rng(0)
noisy_groups = perturb_group(true_groups, drop_frac=0.4, add_frac=0.1, rng=rng)
overlap = int(((true_groups == 1) & (noisy_groups == 1)).sum())
print(f"true members={int(true_groups.sum())}, noisy group members={int(noisy_groups.sum())}, "
      f"correct overlap={overlap}")

In [ ]:
builders_noisy = [
    ElasticNetBuilder(),
    BayesBuilder(prior="regularized_horseshoe", p0=10, num_warmup=300, num_samples=300),
    BayesBuilder(prior="group_horseshoe", p0=10, num_warmup=300, num_samples=300,
                 hyper={"groups": noisy_groups}),
]
res_noisy = run_benchmark(builders_noisy, ds, outer_k=5, seed=0)
summary_table(res_noisy)

In [ ]:
plot_comparison(res_noisy)

## Reading the result

With the perfectly informative group, `group_horseshoe` should out-perform the plain
horseshoe by concentrating posterior mass on the causal genes' shared scale. As the group
gets corrupted (`drop_frac`/`add_frac` above), that advantage should shrink — try pushing
them higher to find the point where an imperfect pathway annotation stops helping at all.
This is the generic version of the eventual use case: a GIP/GLP-1 pathway gene list is never
perfectly known, so this notebook's second half is the relevant robustness check before
trusting a group prior built from a real pathway database.